<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


In [4]:
import os, re, warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer


train_df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_df  = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')

# bucket intensity into 3 classes instead of 5
# tried 5-class first, all models were near random (21-24%), not worth it
train_df["intensity_3"] = train_df["intensity"].map({1:0, 2:0, 3:1, 4:2, 5:2})

STATES = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless"]

# built these vocab lists by going through the actual entries manually
# not perfect but covers most of the meaningful signal
EVOCAB = {
    "calm":        ["calm","settle","settled","quiet","peaceful","lighter","ease","grounded",
                    "slow","soft","slowed","soften","serene","centered","breathe","breath",
                    "float","release","stillness","pause","relief","less tense"],
    "restless":    ["restless","jumpy","racing","fidgety","scattered","distracted","buzz",
                    "switch","bounce","itchy","unable","wander","still busy","mind jumping",
                    "low buzz","keep wanting"],
    "focused":     ["focus","focused","clear","plan","organize","prioritize","lock",
                    "concentrate","ready","tackle","start","clarity","locked in","sharp",
                    "sharper","next steps"],
    "overwhelmed": ["overwhelmed","overloaded","heavy","pressure","carrying","flooded","piled",
                    "drained","everything","behind","hard","exhausted","too much","drowning",
                    "emotionally tired","want to stop"],
    "neutral":     ["normal","same","steady","average","fine","okay","nothing","fairly",
                    "neutral","aware","not much different","mostly same","just normal","no change"],
    "mixed":       ["mixed","split","between","both","part","two","comforted","distracted",
                    "uneasy","lingering","conflicted","pulled","still uneasy","better but",
                    "not fully","two moods"]
}
VMAP = {m: set(v) for m, v in EVOCAB.items()}

FACES    = ["calm_face","happy_face","neutral_face","tired_face","tense_face","none",""]
PMOODS   = ["calm","focused","mixed","neutral","overwhelmed","restless","","none"]
AMBIENCE = ["ocean","forest","mountain","rain","cafe"]
TIMES    = ["morning","afternoon","evening","night","early_morning"]


def face_mood_vec(face, prev):
    face = str(face).strip().lower() if pd.notna(face) else "none"
    prev = str(prev).strip().lower() if pd.notna(prev) else ""
    fv = np.zeros(len(FACES),  dtype=np.float32)
    pv = np.zeros(len(PMOODS), dtype=np.float32)
    for i, fe in enumerate(FACES):
        if fe == face: fv[i] = 1.0; break
    for i, pm in enumerate(PMOODS):
        if str(pm).lower() == prev: pv[i] = 1.0; break
    return np.concatenate([fv, pv])


def sem_sim(j):
    # cosine-ish overlap between journal tokens and each emotion cluster
    if not isinstance(j, str): j = ""
    tok = set(re.findall(r'\b\w+\b', j.lower()))
    return np.array([
        len(tok & v) / (np.sqrt(len(tok) + 1) * np.sqrt(len(v) + 1))
        for v in VMAP.values()
    ], dtype=np.float32)


def ambience_proximity(j, a):
    # if the ambience word shows up in the journal, emotion keywords near it get boosted
    # e.g. "ocean audio was nice" -> calm keywords near "ocean" get 2x weight
    if not isinstance(j, str): j = ""
    if not isinstance(a, str): a = ""
    jl, al = j.lower(), a.lower()
    toks   = re.findall(r'\b\w+\b', jl)
    apos   = [i for i, t in enumerate(toks) if t == al]
    scores = []
    for kws in EVOCAB.values():
        s = 0.0
        for kw in kws:
            if kw in jl:
                base = 1.0
                if apos:
                    kp = [i for i, t in enumerate(toks) if t == kw.split()[0]]
                    for ap in apos:
                        for k in kp:
                            base = max(base, 2.0 / (1 + abs(ap - k) * 0.1))
                s += base
        scores.append(s)
    tot = sum(scores) + 1e-9
    return np.array([s / tot for s in scores] + [float(al in jl)], dtype=np.float32)


def onehot(val, cats):
    v = np.zeros(len(cats), dtype=np.float32)
    s = str(val).lower().strip() if pd.notna(val) else ""
    for i, c in enumerate(cats):
        if c == s: v[i] = 1.0; break
    return v


def safe_float(row, col, fallback=3.0):
    val = row.get(col, fallback)
    try: return float(val) if pd.notna(val) else fallback
    except: return fallback


def build_features(df):
    out = []
    for _, row in df.iterrows():
        j   = row.get("journal_text", "")
        a   = row.get("ambience_type", "")
        ss  = sem_sim(j)
        dom = np.zeros(6, dtype=np.float32); dom[np.argmax(ss)] = 1.0
        srt = np.sort(ss)[::-1]
        num = np.array([
            safe_float(row, "duration_min", 15),
            safe_float(row, "sleep_hours",   6),
            safe_float(row, "energy_level",  3),
            safe_float(row, "stress_level",  3),
        ], dtype=np.float32)
        rq_map = {"vague": 0., "conflicted": .5, "clear": 1.}
        rq = rq_map.get(str(row.get("reflection_quality", "vague")).lower().strip(), .25)
        out.append(np.concatenate([
            ss,
            ambience_proximity(j, a),
            face_mood_vec(row.get("face_emotion_hint", ""), row.get("previous_day_mood", "")),
            onehot(a, AMBIENCE),
            onehot(row.get("time_of_day", ""), TIMES),
            np.array([rq], dtype=np.float32),
            dom,
            np.array([srt[0] - srt[1]], dtype=np.float32),
            num,
        ]))
    return np.array(out, dtype=np.float32)


def row_to_text(row):
    # append ambience/face/mood as tokens so tfidf picks them up as context
    j = str(row.get("journal_text", "")) if pd.notna(row.get("journal_text", "")) else ""
    return f"{j} {row.get('ambience_type','')} {row.get('face_emotion_hint','')} {row.get('previous_day_mood','')}"


# ------- features -------

print("building features...")

tr_texts = [row_to_text(r) for _, r in train_df.iterrows()]
te_texts = [row_to_text(r) for _, r in test_df.iterrows()]

# word bigrams + char ngrams, fit on train only
tw = TfidfVectorizer(ngram_range=(1, 2), max_features=500, sublinear_tf=True, min_df=3)
tc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), max_features=200, sublinear_tf=True, min_df=4)

Tw = tw.fit_transform(tr_texts).toarray(); Tw_te = tw.transform(te_texts).toarray()
Tc = tc.fit_transform(tr_texts).toarray(); Tc_te = tc.transform(te_texts).toarray()

# compress with SVD - 500 sparse cols -> 40 dense, 200 -> 20
sw = TruncatedSVD(min(40, Tw.shape[1] - 1), random_state=42)
sc = TruncatedSVD(min(20, Tc.shape[1] - 1), random_state=42)
Tw = sw.fit_transform(Tw); Tw_te = sw.transform(Tw_te)
Tc = sc.fit_transform(Tc); Tc_te = sc.transform(Tc_te)

Xs    = build_features(train_df)
Xs_te = build_features(test_df)

X_raw    = np.hstack([Tw, Tc, Xs])
X_raw_te = np.hstack([Tw_te, Tc_te, Xs_te])

imp = SimpleImputer(strategy='median')
sca = StandardScaler()
X    = sca.fit_transform(imp.fit_transform(X_raw))
X_te = sca.transform(imp.transform(X_raw_te))

le = LabelEncoder(); le.classes_ = np.array(STATES)
y1 = le.transform(train_df["emotional_state"].str.lower().str.strip())
y2 = train_df["intensity_3"].values

print(f"feature matrix: {X.shape}")

X_tr, X_val, y1_tr, y1_val, y2_tr, y2_val = train_test_split(
    X, y1, y2, test_size=0.15, random_state=42, stratify=y1
)


# ------- 3-fold CV to get a real estimate before committing -------

print("\nrunning 3-fold CV...")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv1, cv2 = [], []

for fold, (tri, vai) in enumerate(skf.split(X, y1)):
    m1 = HistGradientBoostingClassifier(max_iter=300, max_depth=4, learning_rate=0.05,
                                         min_samples_leaf=10, random_state=42, class_weight="balanced")
    m2 = HistGradientBoostingClassifier(max_iter=300, max_depth=3, learning_rate=0.05,
                                         min_samples_leaf=15, random_state=42)
    m1.fit(X[tri], y1[tri]); m2.fit(X[tri], y2[tri])
    cv1.append(m1.score(X[vai], y1[vai]))
    cv2.append(m2.score(X[vai], y2[vai]))
    print(f"  fold {fold+1}  ->  Y1: {cv1[-1]:.3f}   Y2: {cv2[-1]:.3f}")

print(f"  mean  ->  Y1: {np.mean(cv1):.3f}   Y2: {np.mean(cv2):.3f}")
print(f"  std   ->  Y1: {np.std(cv1):.3f}    Y2: {np.std(cv2):.3f}")


# ------- train final models -------

print("\ntraining...")

# Y1 - emotional state
# depth 4 + min_leaf 10 was the sweet spot, deeper ones overfit fast on 1200 samples
clf_emotion = HistGradientBoostingClassifier(
    max_iter=300, max_depth=4, learning_rate=0.05,
    min_samples_leaf=10, random_state=42, class_weight='balanced'
)
clf_emotion.fit(X_tr, y1_tr)
print(f"  emotion val acc : {clf_emotion.score(X_val, y1_val):.3f}")

# Y2 - intensity (3 buckets: low/med/high)
# shallower than Y1 because the signal here is weaker
clf_intensity = HistGradientBoostingClassifier(
    max_iter=300, max_depth=3, learning_rate=0.05,
    min_samples_leaf=15, random_state=42
)
clf_intensity.fit(X_tr, y2_tr)
print(f"  intensity val acc: {clf_intensity.score(X_val, y2_val):.3f}  (3-class, random = 0.33)")


# ------- eval -------

print("\n--- Emotional State ---")
print(classification_report(y1_val, clf_emotion.predict(X_val), target_names=STATES, zero_division=0))

print("--- Intensity (low / medium / high) ---")
print(classification_report(y2_val, clf_intensity.predict(X_val),
      target_names=["low", "medium", "high"], zero_division=0))


# ------- recommendation engine -------
# attention mechanism: Q = predicted probs + intensity, K = template embeddings
# score each template, pick the highest - no branching at all

RECS = [
    {"label": "deep_work",
     "kw": ["focused", "calm", "organized", "clear"],
     "fn": lambda a, t, d, s: f"Your mind is in a receptive state. Use this for deep work. The {a} ambience supported focus today. Start with the hardest task while this clarity holds."},
    {"label": "gentle_reset",
     "kw": ["calm", "settled", "lighter", "peaceful"],
     "fn": lambda a, t, d, s: f"You've settled into a quieter headspace. The {a} ambience anchored this. A short pause or light movement will carry it further into {t}."},
    {"label": "grounding_practice",
     "kw": ["restless", "jumpy", "scattered", "racing"],
     "fn": lambda a, t, d, s: f"Your system is still running fast. The {a} sounds can anchor you - sync your breath slowly. One task at a time will bring the buzz down."},
    {"label": "emotional_offload",
     "kw": ["overwhelmed", "heavy", "flooded", "pressure"],
     "fn": lambda a, t, d, s: f"You're carrying a lot right now. The {a} ambience has been doing quiet work. Try a journal dump or short walk before returning to demands."},
    {"label": "dual_awareness",
     "kw": ["mixed", "split", "between", "uneasy"],
     "fn": lambda a, t, d, s: f"Two emotional currents are running. The {a} setting softened the gap. Don't force resolution - one anchor task will pull you forward."},
    {"label": "steady_continuity",
     "kw": ["neutral", "steady", "same", "fine"],
     "fn": lambda a, t, d, s: f"Your baseline is stable. The {a} session kept things even. Good state for routine work or small creative steps during {t}."},
    {"label": "rest_recovery",
     "kw": ["tired", "tired_face", "drained", "exhausted"],
     "fn": lambda a, t, d, s: f"Fatigue is present. The {a} soundscape offered some softening. Sleep and genuine stillness are the highest-return action right now."},
    {"label": "high_intensity",
     "kw": ["tense_face", "tense", "wound", "unable"],
     "fn": lambda a, t, d, s: f"Physical tension is elevated. Use {a} as a reset between tasks. Break obligations into smaller steps to reduce the felt pressure."},
]

KDIM = len(STATES) + 1  # 7


def make_key(kws):
    k  = np.zeros(KDIM, dtype=np.float32)
    sm = {s: i for i, s in enumerate(STATES)}
    for kw in kws:
        for state, idx in sm.items():
            if kw in state or state in kw or kw in VMAP.get(state, set()):
                k[idx] += 1.0
        if kw in ["tired", "tense", "tense_face", "exhausted", "drained"]:
            k[-1] += 1.0
    return k / (np.linalg.norm(k) + 1e-9)


KMAT = np.array([make_key(r["kw"]) for r in RECS])


def recommend(probs, intensity_bucket, amb, tod, dur, sleep):
    Q    = np.append(probs, intensity_bucket / 2.0).astype(np.float32)
    attn = np.exp(KMAT @ Q / np.sqrt(KDIM))
    attn /= attn.sum()
    rec  = RECS[int(np.argmax(attn))]
    return {
        "recommendation":    rec["fn"](amb, tod, dur, sleep),
        "template":          rec["label"],
        "duration_min":      int(dur),
        "sleep_rec":         8 if sleep < 6 else round(sleep, 1),
        "time_of_day":       tod,
        "attn_weights":      {r["label"]: float(w) for r, w in zip(RECS, attn)},
    }


# ------- run on test set -------

print("\nrunning on test set...")

probs_te  = clf_emotion.predict_proba(X_te)
preds_te  = np.argmax(probs_te, axis=1)
int_te    = clf_intensity.predict(X_te)

state_labels = le.classes_[preds_te]
int_labels   = ["low", "medium", "high"]

rows = []
for i, row in test_df.iterrows():
    idx  = i - test_df.index[0]
    amb  = str(row.get("ambience_type", "")).lower()
    tod  = str(row.get("time_of_day", "")).lower()
    dur  = float(row.get("duration_min", 10))
    slp  = float(row.get("sleep_hours", 7)) if pd.notna(row.get("sleep_hours")) else 7.0
    rec  = recommend(probs_te[idx], int_te[idx], amb, tod, dur, slp)
    rows.append({
        "id":               row["id"],
        "emotional_state":  state_labels[idx],
        "intensity_bucket": int_labels[int_te[idx]],
        "recommendation":   rec["recommendation"],
        "template":         rec["template"],
        "duration_min":     rec["duration_min"],
        "sleep_rec":        rec["sleep_rec"],
        "time_of_day":      rec["time_of_day"],
        "attn_weights":     rec["attn_weights"],
        "confidence":       round(float(probs_te[idx].max()), 3),
        "ambience":         row.get("ambience_type", ""),
    })

building features...
feature matrix: (1200, 110)

running 3-fold CV...
  fold 1  ->  Y1: 0.522   Y2: 0.390
  fold 2  ->  Y1: 0.550   Y2: 0.383
  fold 3  ->  Y1: 0.500   Y2: 0.438
  mean  ->  Y1: 0.524   Y2: 0.403
  std   ->  Y1: 0.020    Y2: 0.024

training...
  emotion val acc : 0.522
  intensity val acc: 0.444  (3-class, random = 0.33)

--- Emotional State ---
              precision    recall  f1-score   support

        calm       0.62      0.62      0.62        32
     focused       0.58      0.48      0.53        29
       mixed       0.47      0.52      0.49        29
     neutral       0.54      0.47      0.50        30
 overwhelmed       0.47      0.55      0.51        29
    restless       0.47      0.48      0.48        31

    accuracy                           0.52       180
   macro avg       0.53      0.52      0.52       180
weighted avg       0.53      0.52      0.52       180

--- Intensity (low / medium / high) ---
              precision    recall  f1-score   suppor